# 04 - EfficientNet Deep Embedding Extraction

This notebook reads saved ECG scalogram images and extracts deep CNN embeddings.

Recommended flow:

1. Start with EfficientNet-B0 to prove the pipeline.
2. Switch to EfficientNet-B4 for the final feature extraction run.
3. Cache embeddings to avoid recomputing them.
4. Export a feature table that can be merged with handcrafted ECG features.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

PROJECT_DIR = Path("..").resolve()
SRC_DIR = PROJECT_DIR / "src"
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(SRC_DIR))

from config import CONFIG
from reproducibility import set_global_seed

seed_state = set_global_seed(CONFIG.seed)
MODEL_NAME = CONFIG.active_model_name
MAX_IMAGES = CONFIG.active_max_records
MANIFEST_PATH = CONFIG.scalogram_manifest_path(MODEL_NAME)
EMBEDDING_DIR = CONFIG.embedding_dir / MODEL_NAME
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH).head(MAX_IMAGES).copy()
print("Images available:", len(manifest))
print("Seed state:", seed_state)
display(manifest.head())


## Dependency Check

This checks whether the deep-learning packages are installed in the active Jupyter kernel. Do not install them on every run; install once, restart the kernel, then continue.


In [ ]:
import importlib.util

missing = [pkg for pkg in ["torch", "torchvision"] if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages:", missing)
    print("Run this once, then restart the kernel:")
    print(f"{sys.executable} -m pip install torch torchvision timm")
else:
    print("Deep learning dependencies are available.")


## Load EfficientNet

This notebook uses `torchvision` first because it is simpler. If you want EfficientNet-B4, install `torch` and `torchvision`, then switch the model name in the cell below.

For quick testing:

- `efficientnet_b0`

For final advisor-requested run:

- `efficientnet_b4`


In [ ]:
import torch
from torch import nn
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if DEVICE == "cpu" else 32
EXPECTED_EMBEDDING_DIM = CONFIG.model_embedding_dim(MODEL_NAME)

if DEVICE == "cpu":
    torch.set_num_threads(4)

if MODEL_NAME == "efficientnet_b0":
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
elif MODEL_NAME == "efficientnet_b4":
    weights = models.EfficientNet_B4_Weights.DEFAULT
    model = models.efficientnet_b4(weights=weights)
else:
    raise ValueError(f"Unsupported model: {MODEL_NAME}")

transform = weights.transforms()

# Remove classifier; output is the pooled CNN embedding.
model.classifier = nn.Identity()
model.eval().to(DEVICE)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("Max images this run:", MAX_IMAGES)
print("Expected embedding dimension:", EXPECTED_EMBEDDING_DIM)
print("Transform:", transform)


## Extract Or Load Cached Embeddings

Each image gets a cached `.npy` embedding. Re-running the notebook will reuse existing embeddings.


In [ ]:
def load_image_tensor(image_path):
    image = Image.open(image_path).convert("RGB")
    return transform(image)


def embedding_path(record_id):
    return EMBEDDING_DIR / f"{record_id}_{MODEL_NAME}.npy"


work_manifest = manifest.copy()
rows = []
cached_count = 0
new_count = 0

for start in tqdm(range(0, len(work_manifest), BATCH_SIZE), desc="Extracting embeddings"):
    batch_df = work_manifest.iloc[start:start + BATCH_SIZE]
    tensors = []
    uncached_rows = []

    for _, row in batch_df.iterrows():
        cache_path = embedding_path(row["record_id"])
        if cache_path.exists():
            embedding = np.load(cache_path)
            rows.append({
                "record_id": row["record_id"],
                "label": row["label"],
                "label_name": row["label_name"],
                "embedding_path": str(cache_path),
                "embedding_dim": int(embedding.shape[-1]),
            })
            cached_count += 1
        else:
            tensors.append(load_image_tensor(row["image_path"]))
            uncached_rows.append(row)

    if tensors:
        batch = torch.stack(tensors).to(DEVICE)
        with torch.no_grad():
            embeddings = model(batch).detach().cpu().numpy()

        for row, embedding in zip(uncached_rows, embeddings):
            if embedding.shape[-1] != EXPECTED_EMBEDDING_DIM:
                raise ValueError(
                    f"{MODEL_NAME} produced {embedding.shape[-1]} features; "
                    f"expected {EXPECTED_EMBEDDING_DIM}."
                )
            cache_path = embedding_path(row["record_id"])
            np.save(cache_path, embedding)
            rows.append({
                "record_id": row["record_id"],
                "label": row["label"],
                "label_name": row["label_name"],
                "embedding_path": str(cache_path),
                "embedding_dim": int(embedding.shape[-1]),
            })
            new_count += 1

embedding_manifest = pd.DataFrame(rows)
embedding_manifest_path = CONFIG.embedding_manifest_path(MODEL_NAME)
embedding_manifest.to_csv(embedding_manifest_path, index=False)

print(f"Saved embedding manifest: {embedding_manifest_path}")
print(f"Loaded from cache: {cached_count:,}")
print(f"Newly extracted: {new_count:,}")
display(embedding_manifest.head())


## Build Deep Feature Table

This converts cached embeddings into a flat CSV table:

`record_id, label, label_name, eff_0000, eff_0001, ...`


In [ ]:
embedding_matrix = np.vstack([
    np.load(path).ravel()
    for path in tqdm(embedding_manifest["embedding_path"], desc="Loading cached embeddings")
])
if embedding_matrix.shape[1] != EXPECTED_EMBEDDING_DIM:
    raise ValueError(
        f"Deep matrix has {embedding_matrix.shape[1]} columns; "
        f"expected {EXPECTED_EMBEDDING_DIM}."
    )

deep_columns = [f"eff_{i:04d}" for i in range(EXPECTED_EMBEDDING_DIM)]
deep_feature_table = pd.concat(
    [
        embedding_manifest[["record_id", "label", "label_name"]].reset_index(drop=True),
        pd.DataFrame(embedding_matrix, columns=deep_columns),
    ],
    axis=1,
)
deep_feature_path = CONFIG.deep_feature_path(MODEL_NAME)
deep_feature_table.to_csv(deep_feature_path, index=False)

print("Deep feature table shape:", deep_feature_table.shape)
print(f"Saved: {deep_feature_path}")
display(deep_feature_table.head())


## Optional: Merge With Handcrafted Features

This prepares the hybrid table. It expects the existing handcrafted CSV to contain `filename` values matching `record_id`.


In [ ]:
HANDCRAFTED_PATH = CONFIG.feature_dir / "enhanced_handcrafted_features.csv"
handcrafted = pd.read_csv(HANDCRAFTED_PATH)

hybrid = handcrafted.merge(
    deep_feature_table.drop(columns=["label_name"]),
    on=["record_id", "label"],
    how="inner",
)

hybrid_path = CONFIG.hybrid_feature_path(MODEL_NAME)
hybrid.to_csv(hybrid_path, index=False)

print("Handcrafted shape:", handcrafted.shape)
print("Deep shape:", deep_feature_table.shape)
print("Hybrid shape:", hybrid.shape)
print(f"Saved: {hybrid_path}")
assert len(hybrid) == len(deep_feature_table), "Hybrid merge dropped records."
assert len([c for c in hybrid.columns if c.startswith("eff_")]) == EXPECTED_EMBEDDING_DIM
display(hybrid.head())
